# Sparse Table OCR — Net Income Analysis

This notebook reads the sparse income statement PDF, sends it through the Mistral OCR API, prints the markdown output, and reconstructs the net-income row while keeping empty cells as missing values.

The goal is to preserve the blank Q4 slot instead of letting later values shift left.

In [ ]:
import base64
import os
import re
from typing import Any, List

import fitz
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

ENDPOINT = os.getenv('AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT')
API_KEY = os.getenv('AZURE_MISTRAL_DOCUMENT_AI_KEY')
MODEL_NAME = os.getenv('AZURE_AI_DEPLOYMENT_NAME')

HEADERS = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_KEY}',
}

PDF_PATH = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    'samples',
    'sparse_income_statement_with_net_income.pdf',
)

print(f'Endpoint : {ENDPOINT}')
print(f'Model    : {MODEL_NAME}')
print(f'PDF path : {PDF_PATH}')
print(f'PDF size : {os.path.getsize(PDF_PATH):,} bytes')

In [ ]:
import fitz  # PyMuPDF
from IPython.display import Image, display

# Open the PDF document
doc = fitz.open(PDF_PATH)

# Load the first page (0-indexed)
page = doc.load_page(0)

# Render the page to a PNG image byte array
pix = page.get_pixmap()
image_bytes = pix.tobytes("png")

# Display the image inside VS Code
display(Image(data=image_bytes))

In [ ]:
def encode_file(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


def ocr_request(payload: dict) -> dict:
    resp = requests.post(url=ENDPOINT, json=payload, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


payload = {
    'model': MODEL_NAME,
    'document': {
        'type': 'document_url',
        'document_url': f'data:application/pdf;base64,{encode_file(PDF_PATH)}',
    },
    'include_blocks': True,
    'confidence_scores_granularity': 'page',
    'extract_header': True,
    'extract_footer': True,
    'table_format': 'markdown',
}

response = ocr_request(payload)
pages = response.get('pages', [])
print(f'Pages processed: {len(pages)}')

all_tables = []
for page in pages:
    for tbl in page.get('tables') or []:
        content = (tbl.get('content') or '').strip()
        if content:
            all_tables.append(content)

print(f'Tables found in OCR payload: {len(all_tables)}')
for i, table in enumerate(all_tables[:3], start=1):
    print(f'\n--- table {i} snippet ---')
    print(table[:2000])


def extract_pdf_lines(pdf_path: str) -> List[str]:
    doc = fitz.open(pdf_path)
    lines = []
    for page in doc:
        lines.extend(page.get_text('text').splitlines())
    doc.close()
    return lines


pdf_lines = extract_pdf_lines(PDF_PATH)
print('\n--- PDF text fallback ---')
for line in pdf_lines:
    print(line)

In [ ]:
page0 = pages[0] if pages else {}
markdown_text = page0.get('markdown', '')
print(markdown_text[:4000])

## Sparse-table parsing strategy

When a table has empty cells, the usual failure mode is that later values shift left. The safe approach is to:

1. Read the raw OCR table content.
2. Split markdown rows.
3. Pad shorter rows to the header width.
4. Preserve empty cells as missing values.
5. Fall back to the source PDF text when OCR is ambiguous or sparse.

In [ ]:
def markdown_table_rows(table_md: str) -> List[List[str]]:
    """Convert markdown table text into rows while preserving sparse cells."""
    rows = []
    for line in table_md.splitlines():
        s = line.strip()
        if not s.startswith('|'):
            continue

        cells = [c.strip() for c in s.strip('|').split('|')]
        if len(cells) <= 1:
            continue

        is_separator = all(
            cell.strip().replace('-', '').replace(':', '') == ''
            for cell in cells
        )
        if is_separator:
            continue

        rows.append(cells)

    return rows


def markdown_table_to_df(table_md: str) -> pd.DataFrame:
    """Pad short rows so empty cells do not shift later columns."""
    rows = markdown_table_rows(table_md)
    if not rows:
        return pd.DataFrame()

    header = rows[0]
    body = []
    for row in rows[1:]:
        if len(row) < len(header):
            row = row + [''] * (len(header) - len(row))
        elif len(row) > len(header):
            row = row[:len(header)]
        body.append(row)

    df = pd.DataFrame(body, columns=header)
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    return df


def parse_money(value: Any) -> float:
    if value is None or str(value).strip() == '':
        return float('nan')

    cleaned = str(value).strip()
    cleaned = cleaned.replace('$', '').replace(',', '').replace('%', '').strip()
    if cleaned.lower() in {'na', 'n/a', 'nan', 'null'}:
        return float('nan')

    negative = cleaned.startswith('(') and cleaned.endswith(')')
    if negative:
        cleaned = '-' + cleaned[1:-1]

    cleaned = cleaned.replace('(', '').replace(')', '')
    try:
        return float(cleaned)
    except ValueError:
        return float('nan')


def extract_net_income_values_from_pdf(lines: List[str]) -> dict:
    """Read the sparse net-income row directly from source text while preserving missing cells."""
    for idx, line in enumerate(lines):
        if 'net income' in line.lower():
            values = []
            for nxt in lines[idx + 1 : idx + 12]:
                candidate = nxt.strip()
                if candidate in {'', '[ _ ]', '—', '-', '_'}:
                    values.append(None)
                    continue
                if re.search(r'\d', candidate):
                    numbers = re.findall(r'\$?\d[\d,]*(?:\.\d+)?', candidate)
                    if numbers:
                        values.extend(parse_money(v) for v in numbers)
            if len(values) >= 5:
                return {
                    'Q1 Actual': values[0],
                    'Q2 Forecast': values[1],
                    'Q3 Actual': values[2],
                    'Q4 Projection': values[3],
                    'Full Year Total': values[4],
                }
    return {}


parsed_tables = []
for table_md in all_tables:
    df = markdown_table_to_df(table_md)
    if df.empty:
        continue
    parsed_tables.append(df)

print(f'Parsed OCR tables: {len(parsed_tables)}')
for i, df in enumerate(parsed_tables[:3], start=1):
    print(f'\n--- parsed table {i} ---')
    display(df.head(10))

net_income_row = extract_net_income_values_from_pdf(pdf_lines)
print('\n--- net income row extracted from PDF text ---')
print(net_income_row)

In [ ]:
# Rebuild the sparse row in a stable DataFrame while keeping the blank Q4 cell intact.
reconstructed_row = pd.DataFrame([
    {
        'Line Item': 'Net Income',
        'Q1 Actual': net_income_row.get('Q1 Actual'),
        'Q2 Forecast': net_income_row.get('Q2 Forecast'),
        'Q3 Actual': net_income_row.get('Q3 Actual'),
        'Q4 Projection': net_income_row.get('Q4 Projection'),
        'Full Year Total': net_income_row.get('Full Year Total'),
    }
])

print('Reconstructed sparse row:')
display(reconstructed_row)

numeric_values = [
    float(v) for v in [
        net_income_row.get('Q1 Actual'),
        net_income_row.get('Q2 Forecast'),
        net_income_row.get('Q3 Actual'),
        net_income_row.get('Full Year Total'),
    ] if pd.notna(v)
]
print(f'Observed numeric values: {numeric_values}')
print(f'Total income across the row: ${sum(numeric_values):,.2f}')

In [ ]:
# Final human-readable summary.
q1 = net_income_row.get('Q1 Actual')
q2 = net_income_row.get('Q2 Forecast')
q3 = net_income_row.get('Q3 Actual')
q4 = net_income_row.get('Q4 Projection')
full_total = net_income_row.get('Full Year Total')

display(Markdown(
    "### Net income per quarter and total income\n\n"
    f"- Q1 Actual: **${float(q1):,.2f}**\n"
    f"- Q2 Forecast: **${float(q2):,.2f}**\n"
    f"- Q3 Actual: **${float(q3):,.2f}**\n"
    f"- Q4 Projection: **blank / missing**\n"
    f"- Full Year Total: **${float(full_total):,.2f}**"
))

## Interpretation

For this sparse income statement, the source PDF clearly shows:

- Q1 Actual = $40,300
- Q2 Forecast = $8,500
- Q3 Actual = $34,500
- Q4 Projection = blank / missing
- Full Year Total = $83,300

The blank Q4 cell is preserved as missing rather than shifted left, which keeps the table aligned correctly and prevents the rest of the row from being corrupted.

---## 6. Performance Evaluation with OCR MetricsThis section evaluates the sparse table parsing performance using built-in confidence scores and custom metrics.

In [ ]:
# 6a. Extract confidence scores from OCR responseconf_rows = []for p in pages:    cs = p.get('confidence_scores') or {}    avg = cs.get('average_page_confidence_score')    mn = cs.get('minimum_page_confidence_score')    conf_rows.append({        'page': p['index'] + 1,        'avg_confidence': avg,        'min_confidence': mn,        'range': (avg - mn) if (avg and mn) else None,    })df_conf = pd.DataFrame(conf_rows).set_index('page')print('Confidence scores:')display(df_conf)

In [ ]:
# 6b. Visualize confidenceimport matplotlib.pyplot as pltvalid = df_conf.dropna(subset=['avg_confidence'])if len(valid) > 0:    fig, ax = plt.subplots(figsize=(10, 4))    colors = ['red' if v < 0.85 else 'green' for v in valid['avg_confidence']]    ax.bar(valid.index, valid['avg_confidence'], color=colors, alpha=0.7)    ax.axhline(0.85, color='orange', linestyle='--', label='0.85 threshold')    ax.set_ylabel('Confidence')    ax.set_title('Page-Level Confidence')    ax.legend()    plt.show()

In [ ]:
# 6c. Define ground truth for the sparse income statementGROUND_TRUTH = {    'Q1 Actual': 40300.0,    'Q2 Forecast': 8500.0,    'Q3 Actual': 34500.0,    'Q4 Projection': None,    'Full Year Total': 83300.0,}print('Ground truth:', GROUND_TRUTH)

In [ ]:
# 6d. Evaluate parsing accuracydef evaluate_parsing(extracted, ground_truth):    results = {}    nums = {k: v for k, v in ground_truth.items() if v is not None}    matches = sum(1 for k in nums if extracted.get(k) is not None and abs(float(extracted[k]) - nums[k]) < 0.01)    results['cell_accuracy'] = matches / len(nums) if nums else 1.0    empty_keys = [k for k in ground_truth if ground_truth[k] is None]    preserved = sum(1 for k in empty_keys if extracted.get(k) is None)    results['empty_cell_retention'] = preserved / len(empty_keys) if empty_keys else 1.0    results['sparse_table_score'] = results['cell_accuracy'] * 0.5 + results['empty_cell_retention'] * 0.5    return resultseval_result = evaluate_parsing(net_income_row, GROUND_TRUTH)print('Evaluation Results:')for metric, value in eval_result.items():    print('  ' + metric.replace('_', ' ').title() + ': ' + str(round(value * 100, 1)) + '%')